In [ ]:
using DelimitedFiles;   #Paquetería para leer y escribir archivos con datos
using CairoMakie;       #Paquetería para realizar gráficas
using FFTW;             #Paquetería para realizar transformadas rápidas de Fourier
using LaTeXStrings;     #Paquetería para usar texto en LaTeX en gráficas
using LsqFit;           #Paquetería para realizar ajuste de mínimos cuadrados
using LombScargle;      #Paquetería para realizar un periodograma (similar a una FFT)
using ConcaveHull;      #Paquetería para generar el Concave Hull de un conjunto de puntos
using Statistics;       #Paquetería para usar funciones básicas en estadística como la media, el promedio, etc.
using LombScargle;      #Paquetería para generar un periodograma de LombScargle

#Definimos la función que calcula nuestro aproximante estadístico
AproxLambda(NSides) = (2π / (1 - cos(2π / NSides)));

#Función que "suaviza" una gráfica reduciendo el número de puntos por su promedio (media móvil)
function promedio_m(A, i, m, Factor)
    Valores = [Factor*A[j][2] for j in (i-m):(i+m)]
    return mean(Valores)
end

DataPath = "Quasiperiodic-Tiles/Global Structural Studies/Data/SI_Fig3-A_SigmaSquare_2D"; #Ruta donde los datos están almacenados

In [ ]:
#Diccionario con los valores del Concave Hull usado para los datos sin el inicio de las g(R)
PCH_Dict = Dict(
                5  => 331,
                7  => 442,
                9  => 50,
                11 => 195,
                13 => 975,
                15 => 272,
                17 => 346,
                19 => 231
               );

#Diccionario con los índices para eliminar el primer gran pico de la g(R)
Start_Index_Dict = Dict(
                        5  => 1,
                        7  => 1,
                        9  => 1002,
                        11 => 1002,
                        13 => 1002,
                        15 => 1002,
                        17 => 1002,
                        19 => 1002
                       );

#Diccionario con los valores de la densidad numérica de los sistemas cuasiperiódicos en 2D
Rho_Dict = Dict(
                5  => 1.2328979808609704,
                7  => 1.2517957581185175,
                9  => 1.260284085456272,
                11 => 1.2645739922427748,
                13 => 1.2670366156186388,
                15 => 1.2685820147323164,
                17 => 1.2696141740269873,
                19 => 1.2703373895977548,
                21 => 1.2708642307351703,
                23 => 1.2712590235307708,
                25 => 1.271563821555834,
                27 => 1.271802352106815,
                29 => 1.2719940413420914,
                31 => 1.2721488757491926
               );

#Diccionario con los valores de la k_N por periodograma de los sistemas cuasiperiódicos en 2D
K_Dict = Dict(
              9  => 25.70401,
              11 => 21.384,
              13 => 24.45278,
              15 => 29.92639,
              17 => 33.06918,
              19 => 37.30495
             );

A_Dict = Dict(
              9  => 0.5114562,
              11 => 0.6651776,
              13 => 0.3342172,
              15 => 0.336498,
              17 => 0.2610448,
              19 => 0.1784393
             );   

# Varianza $\sigma^2(R)$

In [ ]:
##########################################################################################################################################################################
#                                                                   Datos del sistema cuasiperiódico
##########################################################################################################################################################################
NSides = 19;                    #Simetría rotacional de los sistemas cuasperiódicos a analizar
Radio = 500;                    #Radio de las vecindades circulares
NumPasos = Int(1e5);            #Número de pasos a dar desde R = 0 hasta R = Radio para variar el radio de la ventana circular de N(R)
ΔStep = Radio/NumPasos;         #Tamaño del salto entre radio y radio
##########################################################################################################################################################################
#                                                                      Datos del lienzo a graficar
##########################################################################################################################################################################
# --- Definición de las características del lienzo y las subgráficas en él ---
Fig = Figure(size = (1800, 800)); #Lienzo en blanco donde se graficara
FFT_Ax = Axis(
              Fig[1, 1],                    #Posición en el lienzo donde se realizará la gráfica
              xlabel = L"R = 2 \pi / k",    #Etiqueta que aparece en el eje horizontal
              ylabel = L"A",                #Etiqueta que aparece en el eje vertical
              titlesize = 55,               #Tamaño del título
              xlabelsize = 55,              #Tamaño de la etiqueta al eje horizontal
              ylabelsize = 55,              #Tamaño de la etiqueta al eje vertical
              xticklabelsize = 40,          #Tamaño para el eje X
              yticklabelsize = 40,          #Tamaño para el eje Y
              xticksize = 25,               #Tamaño de los ticks horizontales
              yticksize = 25,               #Tamaño de los ticks verticales
              limits = (nothing, nothing),  #Límites de la visualización para la gráfica
              xscale = log10,
              yscale = log10
             );
hidespines!(FFT_Ax, :t, :r);                #Remueve las líneas de la caja que rodea a la gráfica ':t' = top, ':r' = right
hidedecorations!(
                 FFT_Ax,
                 label = false,             #Se oculta o no las etiquetas a los ejes
                 ticklabels = false,        #Se oculta o no los valores de los ticks de los ejes
                 ticks = false              #Se oculta o no los ticks de los ejes
                );
##########################################################################################################################################################################
#                                                                      Iteración de casos
##########################################################################################################################################################################
NSides_Array = [5, 9, 13, 17, 21, 25, 29];                                      #Arreglo de las simetrías a graficar
Paleta_Colores = cgrad(:inferno, length(NSides_Array) + 1, categorical = true); #Definimos el gradiente de colores para la gráfica
λ_N_Array = [];                                                                 #Arreglo que contendrá las longitudes de escala λ_N
Amp_λ_N_Array = [];                                                             #Arreglo que contendrá las amplitudes de las λ_N
for i in 1:length(NSides_Array)
    NSides = NSides_Array[i];
    ##########################################################################################################################################################################
    #                                                                  Lectura de los datos de la Sigma^2
    ##########################################################################################################################################################################
    σ2 = vec(readdlm(DataPath * "Torquato_NR_N$(NSides)_Alfa0P0_R$(Radio)_Step1e5_SigmaCuadrada.csv"));
    ##########################################################################################################################################################################
    #                                                  Cálculo de la densidad promedio y factor de normalización (Torquato)
    ##########################################################################################################################################################################
    Rho = Rho_Dict[NSides];         #Densidad numérica de sitios a través del diccionario
    FN = 2*sqrt(π*Rho);             #Factor de normalización para mantener los resultados independientes de la densidad de puntos
    ##########################################################################################################################################################################
    #                                                           Cálculo del arreglo de radios para la sigma^2
    ##########################################################################################################################################################################
    #Generamos el intervalo con los valores de la R asociados a los datos de sigma cuadrada (Incluye factor de 2*sqrt(π*Rho)
    #necesario para mantener densidad de puntos constantes en decorado, independientemente de la simetría rotacional)
    R = ΔStep:ΔStep:Radio;
    R = FN .* R;
    ##########################################################################################################################################################################
    #                                                           Cálculo teórico de la longitud de escala λ_N
    ##########################################################################################################################################################################
    λ = AproxLambda(NSides);        #Longitud de escala de nuestro sistema
    ##########################################################################################################################################################################
    #                                                                       Cálculo de la FFT
    ##########################################################################################################################################################################
    σ2R = σ2./R;                    #Valores en el eje vertical para obtener su FFT
    N = length(σ2R);                #Número de datos a analizar
    νs = fftfreq(N, 1/(FN*ΔStep));  #Rango de frecuencias que obtiene la transformada de Fourier
    n = floor(Int, length(νs)/2);
    X2 = fft(σ2R);                  #Transformada de Fourier (está en los complejos, así que hay que normalizar con abs)
    ##########################################################################################################################################################################
    #                                                          Cálculo empírico de la longitud de escala λ_N
    ##########################################################################################################################################################################
    λs = 1 ./ νs[2:1:n];            #Inverso de las frecuencias espaciales
    FT = abs.(X2)[2:1:n];           #Transformada de Fourier de la gráfica, normalizada con abs
    AP, IndiceAP = findmax(FT);     #Amplitud Principal de la Transformada de Fourier y su correspondiente índice en el arreglo
    λAP = λs[IndiceAP];             #Longitud asociada a la Amplitud Principal
    println("La longitud de escala λ = $(λAP)");
    push!(λ_N_Array, λAP);
    push!(Amp_λ_N_Array, FT[IndiceAP]);
    ##########################################################################################################################################################################
    #                                                         Gráfica de la FFT con su longitud de escala λ_N
    ##########################################################################################################################################################################
    # --- Gráfica de los datos de la FFT ---
    lines!(
           FFT_Ax, λs, (10^(3*(i-1))).*FT,
           color = (Paleta_Colores[i], 1)
          );
end
###################################################################################################################
#                                            Gráfica del máximo de la transformada de Fourier    
###################################################################################################################
for i in 1:length(NSides_Array)
    scatter!(
             FFT_Ax, λ_N_Array[i], (10^(3*(i-1))).*Amp_λ_N_Array[i],
             color = :red,
             markersize = 20
            );
end
###################################################################################################################
#                                            Guardamos las gráficas    
###################################################################################################################
Fig

### Comparativa de la longitud característica de FFTW con nuestra expresión teórica

In [ ]:
#Valores de las simetrías rotacionales analizadas
N_Array = [5, 7, 9, 11, 13, 15, 17, 19, 21, 23, 25, 27, 29, 31];
#Valores de las λ_N asociadas a las simetrías rotacionales analizadas
λ_N_Array = [
             1.8797142133748708,
             1.7139901239195763,
             25.841542726032934,
             38.3304321406405,
             53.92222386035155,
             71.29780832456713,
             90.7795732158813,
             117.51290307922093,
             142.72382584636222,
             166.53699140994178,
             199.86834567710747,
             222.0967680933104,
             285.5745063684034,
             333.19053450966805
            ];

# --- Definición de las características del lienzo y las subgráficas en él ---
Fig = Figure(size = (1800, 800)); #Lienzo en blanco donde se graficara
λ_Comparison_Ax = Axis(
                       Fig[1, 1],                                                   #Posición en el lienzo donde se realizará la gráfica
                       #title = L"N = %$(NSides)",                                  #Título de la gráfica
                       xlabel = L"N",                                               #Etiqueta que aparece en el eje horizontal
                       ylabel = L"\lambda_{N}",                                     #Etiqueta que aparece en el eje vertical
                       titlesize = 55,                                              #Tamaño del título
                       xlabelsize = 55,                                             #Tamaño de la etiqueta al eje horizontal
                       ylabelsize = 55,                                             #Tamaño de la etiqueta al eje vertical
                       xticklabelsize = 40,                                         #Tamaño para el eje X
                       yticklabelsize = 40,                                         #Tamaño para el eje Y
                       xticksize = 25,                                              #Tamaño de los ticks horizontales
                       yticksize = 25,                                              #Tamaño de los ticks verticales
                       limits = (nothing, nothing),                                 #Límites de la visualización para la gráfica
                       #xscale = log10,
                       #yscale = log10
                      );
hidespines!(λ_Comparison_Ax, :t, :r); #Remueve las líneas de la caja que rodea a la gráfica ':t' = top, ':r' = right
hidedecorations!(
                 λ_Comparison_Ax,
                 label = false,           #Se oculta o no las etiquetas a los ejes
                 ticklabels = false,      #Se oculta o no los valores de los ticks de los ejes
                 ticks = false            #Se oculta o no los ticks de los ejes
                )
# --- Gráfica de los datos de la FFT ---
lines!(λ_Comparison_Ax, N_Array, AproxLambda.(N_Array));
scatter!(
         λ_Comparison_Ax, N_Array, λ_N_Array,
         color = :red,
         markersize = 15
        );
###################################################################################################################
#                                            Guardamos las gráficas    
###################################################################################################################
Fig